# Match-TTSG: UNIFIED SPEECH AND GESTURE SYNTHESIS USING FLOW MATCHING
---
[Shivam Mehta](https://www.kth.se/profile/smehta), [Ruibo Tu](https://www.kth.se/profile/ruibo), [Simon Alexanderson
](https://www.kth.se/profile/simonal?l=en) [Jonas Beskow](https://www.kth.se/profile/beskow), [Éva Székely](https://www.kth.se/profile/szekely), and [Gustav Eje Henter](https://people.kth.se/~ghe/)

As text-to-speech technologies achieve remarkable naturalness in read-aloud tasks, there is growing interest in multimodal synthesis of verbal and non-verbal communicative behaviour, such as spontaneous speech and associated body gestures. This paper presents a novel, unified architecture for jointly synthesising speech acoustics and skeleton-based 3D gesture motion from text, trained using optimal-transport conditional flow matching (OT-CFM). The proposed architecture is simpler than the previous state of the art, has a smaller memory footprint, and can capture the joint distribution of speech and gestures, generating both modalities together in one single process. The new training regime, meanwhile, enables better synthesis quality in much fewer steps (network evaluations) than before. Uni- and multimodal subjective tests demonstrate improved speech naturalness, gesture human-likeness, and cross-modal appropriateness compared to existing benchmarks.

Demo Page: https://shivammehta25.github.io/Match-TTSG \
Code: https://github.com/shivammehta25/Match-TTSG




In [36]:
%env CUDA_VISIBLE_DEVICES=0

env: CUDA_VISIBLE_DEVICES=0


In [37]:
import datetime as dt
import warnings
from pathlib import Path

import ffmpeg
import IPython.display as ipd
import joblib as jl
import numpy as np
import soundfile as sf
import torch
from tqdm.auto import tqdm

# Hifigan imports
from match_ttsg.hifigan.config import v1
from match_ttsg.hifigan.denoiser import Denoiser
from match_ttsg.hifigan.env import AttrDict
from match_ttsg.hifigan.models import Generator as HiFiGAN
# MatchTTSG imports
from match_ttsg.models.match_ttsg_custom_deca import MatchTTSGCustomDECA
from match_ttsg.text import sequence_to_text, text_to_sequence
from match_ttsg.utils.model import denormalize
from match_ttsg.utils.utils import get_user_data_dir, intersperse

# Motion visualisation imports
from pymo.preprocessing import MocapParameterizer
from pymo.viz_tools import render_mp4
from pymo.writers import BVHWriter

from speechbrain.inference.vocoders import HIFIGAN
import pandas as pd
import pickle

In [38]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
# This allows for real time code changes being reflected in the notebook, no need to restart the kernel

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

True


## Filepaths

In [40]:
MATCHTTSGDECA_CHECKPOINT = "/home/git/Match-TTSG/logs/train/multimodal_base_verbatim_baseline/runs/2025-09-11_14-47-30/checkpoints/last.ckpt"
#MOTION_PIPELINE = "match_ttsg/resources/data_pipe.expmap_86.1328125fps.sav"
HIFIGAN_CHECKPOINT = get_user_data_dir() / "generator_v1"#"hifigan_T2_v1"
OUTPUT_FOLDER = "synth_output"

print(HIFIGAN_CHECKPOINT)

/home/ayrton/.local/share/matcha_tts/generator_v1


## Load Match-TTSG

In [41]:
def load_model(checkpoint_path):
    model = MatchTTSGCustomDECA.load_from_checkpoint(checkpoint_path, map_location=device)
    model.eval()
    return model
count_params = lambda x: f"{sum(p.numel() for p in x.parameters()):,}"


model = load_model(MATCHTTSGDECA_CHECKPOINT)
print(f"Model loaded! Parameter count: {count_params(model)}")

Model loaded! Parameter count: 18,453,009


## Load HiFi-GAN (Vocoder)

In [42]:
def load_vocoder(checkpoint_path):
    h = AttrDict(v1)
    hifigan = HiFiGAN(h).to(device)
    hifigan.load_state_dict(torch.load(checkpoint_path, map_location=device)['generator'])
    _ = hifigan.eval()
    hifigan.remove_weight_norm()
    return hifigan

vocoder = load_vocoder(HIFIGAN_CHECKPOINT)
#denoiser = Denoiser(vocoder, mode='zeros')

Removing weight norm...


/tmp/ipykernel_4060136/984558455.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hifigan.load_state_dict(torch.load(checkpoint_path, map_location=device)['generator'])


# Load motion visualiser

In [43]:
## Number of ODE Solver steps
n_timesteps = 500

## Changes to the speaking rate
length_scale=1.0

## Sampling temperature
temperature = 0.667

### Helper functions to synthesise

In [44]:
@torch.inference_mode()
def process_text(text: str):
    #x = torch.tensor(intersperse(text_to_sequence(text, ['japanese_accent_cleaners']), 0),dtype=torch.long, device=device)[None]
    x = torch.tensor(text_to_sequence(text, ['japanese_accent_cleaners']),dtype=torch.long, device=device)[None]
    x_lengths = torch.tensor([x.shape[-1]],dtype=torch.long, device=device)
    x_phones = sequence_to_text(x.squeeze(0).tolist())
    return {
        'x_orig': text,
        'x': x,
        'x_lengths': x_lengths,
        'x_phones': x_phones
    }


@torch.inference_mode()
def synthesise(text, spks=None):
    text_processed = process_text(text)
    start_t = dt.datetime.now()
    output = model.synthesise(
        text_processed['x'], 
        text_processed['x_lengths'],
        n_timesteps=n_timesteps,
        temperature=temperature,
        spks=spks,
        length_scale=length_scale
    )
    # merge everything to one dict    
    output.update({'start_t': start_t, **text_processed})
    return output

@torch.inference_mode()
def to_waveform(mel, vocoder):
    hifi_gan = HIFIGAN.from_hparams(source="speechbrain/tts-hifigan-libritts-16kHz", savedir="pretrained_models/tts-hifigan-libritts-16kHz")
    audio = hifi_gan.decode_batch(mel)
    return audio.cpu().squeeze()

"""
def to_waveform(mel, vocoder):
    audio = vocoder(mel).clamp(-1, 1)
    audio = denoiser(audio.squeeze(0), strength=0.00025).cpu().squeeze()
    return audio.cpu().squeeze()
"""

"""
def to_bvh(motion):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return motion_pipeline.inverse_transform([motion.cpu().squeeze(0).T])
"""  
def save_to_folder(filename: str, output: dict, folder: str):
    folder = Path(folder)
    folder.mkdir(exist_ok=True, parents=True)
    np.save(folder / f'{filename}', output['mel'].cpu().numpy())
    sf.write(folder / f'{filename}.wav', output['waveform'], 16000, 'PCM_24')
    #with open(folder / f'{filename}.bvh', 'w') as f:
        #bvh_writer.write(output['bvh'], f)

def save_to_movement_csv(filename: str, output: dict, folder:str):
    folder = Path(folder)
    folder.mkdir(exist_ok=True, parents=True)
    expression_df = pd.DataFrame(output['expression'].cpu().squeeze().T.numpy())
    pose_df = pd.DataFrame(output['pose'].cpu().squeeze().T.numpy())
    with open(f"./{folder}/{filename}.decaexp.pkl", 'wb') as f:
        pickle.dump(expression_df, f)
    with open(f"./{folder}/{filename}.decapose.pkl", 'wb') as f:
        pickle.dump(pose_df, f)
    
"""
def to_stick_video(filename, bvh, folder):
    folder = Path(folder)
    folder.mkdir(exist_ok=True, parents=True)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        X_pos = mocap_params.fit_transform([bvh])
    print(f"rendering {filename} ...")
    render_mp4(X_pos[0], folder / f'{filename}.mp4', axis_scale=200)


def combine_audio_video(filename: str, folder: str):
    print("Combining audio and video")
    folder = Path(folder)
    folder.mkdir(exist_ok=True, parents=True)

    input_video = ffmpeg.input(str(folder / f'{filename}.mp4'))
    input_audio = ffmpeg.input(str(folder / f'{filename}.wav'))
    output_filename = folder / f'{filename}_audio.mp4'
    ffmpeg.concat(input_video, input_audio, v=1, a=1).output(str(output_filename)).run(overwrite_output=True)
    print(f"Final output with audio: {output_filename}")
"""

'\ndef to_stick_video(filename, bvh, folder):\n    folder = Path(folder)\n    folder.mkdir(exist_ok=True, parents=True)\n\n    with warnings.catch_warnings():\n        warnings.simplefilter("ignore")\n        X_pos = mocap_params.fit_transform([bvh])\n    print(f"rendering {filename} ...")\n    render_mp4(X_pos[0], folder / f\'{filename}.mp4\', axis_scale=200)\n\n\ndef combine_audio_video(filename: str, folder: str):\n    print("Combining audio and video")\n    folder = Path(folder)\n    folder.mkdir(exist_ok=True, parents=True)\n\n    input_video = ffmpeg.input(str(folder / f\'{filename}.mp4\'))\n    input_audio = ffmpeg.input(str(folder / f\'{filename}.wav\'))\n    output_filename = folder / f\'{filename}_audio.mp4\'\n    ffmpeg.concat(input_video, input_audio, v=1, a=1).output(str(output_filename)).run(overwrite_output=True)\n    print(f"Final output with audio: {output_filename}")\n'

## Setup text to synthesise

In [45]:
test_data_text_list = []
test_data_text_path = "../data/multimodal_single_kon_verbatim_DECA/test.txt"
with open(test_data_text_path, 'r') as f:
    for line in f:
        text = line.strip().split("|")[1]
        print(text)
        test_data_text_list.append(text)

水泳とか、機械体操とか、そういうのは、まあ、短距離とかは、まあ、できるんですけど、もう、物を使うってなったと、途端に、もう。
当てて、しかも追っかけてって、また当てて追っかけてってって、あれ、なんか、
見る専門でお願いしますって感じですね。そこ、あの、応援しますみたいな感じですね。
さ、じゃあ、えっと、今日のテーマはスポーツについてですが、
うーん、なんかもう、ちょっと、その、応援するっていうか、まあ、確かに、ド、まあ、ドラマがあるので、それを、ドラマを、人間ドラマを見るみたいな感じで観戦はしたりはしますけど。
ねー、ちょっと。
気持ち、楽しいって言ったらやっぱり対戦の方が、できなくても楽しいのは対戦の方かなと思いますよね。
そういうものだったんだ。
よくわかんないです。なんか、私の中では、あの、鉄線とかのイメージ。えへへ。なぜ、なんでそんな鉄線とか貼っちゃうのかなーみたいな。刺さるし。
美しい人とかいるじゃないですか、なんか。いますよね、その、戦いが美しい人が。
特にね、夏はビールを片手にね、観戦するっていうのが最高です。
あれですかね、フットサルってあの、サッカーよりもコートが狭いんでしたっけ?
体育館の半分ぐらいな感じの。
うん。
そこか。そういう理由。
今も、え、なに、何県ですか?
よろしくお願いします。えーと、今回のテーマは動物ということなんですが、どうですか?あの、動物はお好きな方ですか?
すっごい失踪してた記憶があります。公園とかを。
そっか、長毛はだから、か、長毛はだから、こう、ただつくじゃなく、からまるー。
そんな子がいるんですか?毛が抜けない品種の。抜けづらい。
くしゃくしゃしてる。くしゃくしゃした系。
なんか、やっぱり、犬は、あれですよね、結構品種改良がしっかりしてるっていう感じですよね。
でも、なんか、猫は、その、飼いやすさとか、働き方とかじゃなく、ただただ可愛さを追求した変種改良ですよね。
しっぽで、あの、ブラッシングすると、すごい嫌がられました。
わかります。毛、毛がね、生えてないの、ちょっとね、はっ、ちょっと、エキゾチックなやつとか。
あれ、なんなんだろうと思って、絶対間に入る。
ちょっとしたくぼみが、こう、心地いいんだと思うんですけど、脇の下とかね。
犬だと、あ、でも、犬って、もう、寝るときは寝室とかにベッド置いたりとかして、一緒に寝

In [46]:
texts = [
    "最近芸人さんとかでそういう方多いですもんね。",
    "コントローラーでこう、手でやるよりも、マウスの方がピタッとくる感じなんですかね。",
    "実は全然、顔を映さなければ、全然別人だったり、みたいな。"
]

## Synthesis

In [ ]:
outputs, rtfs = [], []
rtfs_w = []
for i, text in enumerate(tqdm(texts)):
    output = synthesise(text) #, torch.tensor([15], device=device, dtype=torch.long).unsqueeze(0))
    print(output['decoder_outputs_mel'].shape)
    output['waveform'] = to_waveform(output['mel'], vocoder)

    # Compute Real Time Factor (RTF) with HiFi-GAN
    t = (dt.datetime.now() - output['start_t']).total_seconds()
    rtf_w = t * 16000 / (output['waveform'].shape[-1])
    dur = output['waveform'].shape[-1] / 16000

    ## Pretty print
    """
    print(f"{'*' * 53}")
    print(f"Input text - {i}")
    print(f"{'-' * 53}")
    print(output['x_orig'])
    print(f"{'*' * 53}")
    print(f"Phonetised text - {i}")
    print(f"{'-' * 53}")
    print(output['x_phones'])
    print(f"{'*' * 53}")
    print(f"RTF:\t\t{output['rtf']*16000/22050:.6f}")
    print(f"RTF Waveform:\t{rtf_w:.6f}")
    print(f"Audio duration:\t{dur:.6f}")
    """
    rtfs.append(output['rtf'])#*16000/22050)
    rtfs_w.append(rtf_w)
    outputs.append(output)

    ## Display the synthesised waveform
    ipd.display(ipd.Audio(output['waveform'], rate=16000))

    ## Save the generated waveform
    #save_to_folder(f"test_base_500_{i}", output, OUTPUT_FOLDER)

    ## face
    #save_to_movement_csv(f"test_base_500_{i}", output, "movement_output")

print(f"Number of ODE steps: {n_timesteps}")
print(f"Mean RTF:\t\t\t\t{np.mean(rtfs):.6f} ± {np.std(rtfs):.6f}")
print(f"Mean RTF Waveform (incl. vocoder):\t{np.mean(rtfs_w):.6f} ± {np.std(rtfs_w):.6f}")

  0%|          | 0/3 [00:00<?, ?it/s]

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-libritts-16kHz' if not cached


torch.Size([1, 80, 166])


INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-libritts-16kHz' if not cached
INFO:speechbrain.utils.fetching:Fetch generator.ckpt: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-libritts-16kHz' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: generator


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-libritts-16kHz' if not cached


...

INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-libritts-16kHz' if not cached
INFO:speechbrain.utils.fetching:Fetch generator.ckpt: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-libritts-16kHz' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: generator


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-libritts-16kHz' if not cached


torch.Size([1, 80, 277])


INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-libritts-16kHz' if not cached
INFO:speechbrain.utils.fetching:Fetch generator.ckpt: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-libritts-16kHz' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: generator


Number of ODE steps: 500
Mean RTF:				1.362724 ± 0.382496
Mean RTF Waveform (incl. vocoder):	1.621884 ± 0.447497


In [48]:
print(f"Number of ODE steps: {n_timesteps}")
print(f"Mean RTF:\t\t\t\t{np.mean(rtfs):.6f} ± {np.std(rtfs):.6f}")
print(f"Mean RTF Waveform (incl. vocoder):\t{np.mean(rtfs_w):.6f} ± {np.std(rtfs_w):.6f}")

Number of ODE steps: 500
Mean RTF:				1.362724 ± 0.382496
Mean RTF Waveform (incl. vocoder):	1.621884 ± 0.447497
